# AI工学101 — 第32回

## 時系列データと交差検証：未来の情報を混ぜない

よし、今日は第31回の**Data Leakage**をさらに実戦的にする回だよ。

これまでの分類では、

```text
全データ
↓
ランダムにtrain/testへ分割
```

をよく使ってきた。

でも、データに**時間の順序**があるとき、この方法は危険になる。

例えば、

```text
2024年
2025年
2026年
```

のデータをランダムに混ぜてしまうと、

> 2026年の情報で学習して、2024年を予測する

ような状況が起こり得る。

実際の本番環境では、

```text
過去
↓
現在
↓
未来を予測
```

なので、評価でもその構造を守る必要がある。

今日は、

```text
過去で学習
↓
未来で評価
```

という時系列機械学習の基本を作る。

---

# 🎯 今日のゴール

* 時系列データでランダム分割が危険な理由を理解する
* `TimeSeriesSplit` を使える
* Walk-forward validationを理解する
* 時系列のData Leakageを見抜ける
* 過去→未来の予測構造を設計できる
* Concept Driftの入口を理解する
* 最終ミニプロジェクトで使える評価設計を1つ増やす

---

# 📖 講義：約20〜25分

## 1. 普通のtrain/test splitが前提としているもの

これまで、

```python
train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

を使ってきた。

これは多くの場合、

> **trainとtestがだいたい同じ分布から来ている**

という前提で便利。

例えば、

```text
犬
猫
犬
猫
犬
猫
```

のような画像分類なら、

```text
ランダム分割
```

が自然な場合が多い。

しかし、

```text
2024年1月
2024年2月
2024年3月
...
2026年
```

のように時間順が意味を持つ場合、

**ランダムシャッフルそのものが問題になる**。

---

# 🧠 2. 何が問題なのか？

例えば、

```text
1月
2月
3月
4月
5月
6月
```

の売上データがあるとする。

ランダム分割すると、

```text
train:
1月
3月
5月
6月

test:
2月
4月
```

のようになる可能性がある。

でも実際には、

```text
1月時点
↓
未来の6月の情報はまだ存在しない
```

よね。

したがって、

> **未来の情報を見ながら過去を評価する**

という不自然な実験になる。

---

# 🧠 3. 時系列の基本構造

時系列では、

```text
過去
↓
学習

未来
↓
評価
```

が基本。

例えば、

```text
2024年 ──── TRAIN

2025年 ──── VALIDATION

2026年 ──── TEST
```

のように分ける。

これは普通のデータ分割より、

**現実の予測状況に近い**。

---

# 💻 実習1：簡単な時系列データを作る

まず人工データを作る。

```python
import numpy as np
import pandas as pd

rng = np.random.RandomState(42)

n = 300

time = np.arange(n)

signal = (
    np.sin(time / 10)
    + time * 0.01
    + rng.normal(0, 0.3, n)
)

df = pd.DataFrame({
    "time": time,
    "value": signal
})

df.head()
```

まず可視化。

```python
import matplotlib.pyplot as plt

plt.plot(
    df["time"],
    df["value"]
)

plt.xlabel("time")
plt.ylabel("value")

plt.show()
```

ここでは、

```text
過去の値
↓
次の値を予測
```

という問題を作る。

---

# 💻 実習2：Lag特徴量

時系列では、

> **過去の値を特徴量にする**

ことが多い。

例えば、

```text
t-1
t-2
t-3
```

の値から、

```text
t
```

を予測する。

Pandasの `shift()` を使う。

```python
df["lag_1"] = df["value"].shift(1)
df["lag_2"] = df["value"].shift(2)
df["lag_3"] = df["value"].shift(3)
```

ターゲットを、

```text
次の時点の値
```

にする。

```python
df["target"] = df["value"].shift(-1)
```

すると、

```text
lag_1
lag_2
lag_3
↓
次の値
```

という予測問題になる。

---

# 🚨 ここで欠損値

`shift()` を使うと、

最初と最後に欠損値ができる。

```python
df = df.dropna()
```

確認。

```python
df.head()
```

---

# 🧠 4. 時系列特徴量の超重要ポイント

ここで、

```text
target = shift(-1)
```

を作ったのは、

> **未来を予測したいから**

です。

一方、

```python
df["future"] = df["value"].shift(-1)
```

を特徴量としてモデルに入れたら？

それは、

```text
未来の値
↓
未来の値を予測
```

していることになる。

つまり、

> **Target Leakage**

そのもの。

---

# 💻 実習3：Xとyを作る

```python
X = df[
    [
        "lag_1",
        "lag_2",
        "lag_3"
    ]
]

y = df["target"]
```

まず確認。

```python
print(X.shape)
print(y.shape)
```

---

# 📖 5. ランダム分割をやってみる

まず、

```python
train_test_split()
```

で分けてみる。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)
```

モデル。

```python
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)
```

評価。

```python
from sklearn.metrics import mean_squared_error

pred = model.predict(X_test)

rmse = mean_squared_error(
    y_test,
    pred
) ** 0.5

print(rmse)
```

ここでRMSEがかなり良く出ても、

> **その評価が未来予測として妥当とは限らない**

。

---

# 💻 実習4：時間順に分割する

今度は、

```text
最初の80%
↓
train

最後の20%
↓
test
```

にする。

```python
split_index = int(
    len(X) * 0.8
)
```

```python
X_train = X.iloc[
    :split_index
]

X_test = X.iloc[
    split_index:
]

y_train = y.iloc[
    :split_index
]

y_test = y.iloc[
    split_index:
]
```

これなら、

```text
過去
↓
train

未来
↓
test
```

になる。

---

# 💻 実習5：同じモデルを評価

```python
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

pred = model.predict(
    X_test
)

rmse = mean_squared_error(
    y_test,
    pred
) ** 0.5

print(
    "RMSE:",
    rmse
)
```

ランダム分割と比較する。

```text
Random split RMSE
vs
Time-based split RMSE
```

を確認しよう。

時間順評価のほうが悪くなることもある。

でも、

**未来に対する性能を見るなら、そちらのほうが意味がある**。

---

# 📖 6. TimeSeriesSplit

ここから今日の主役。

scikit-learnには、

```python
TimeSeriesSplit
```

がある。

```python
from sklearn.model_selection import TimeSeriesSplit
```

例えば、

```python
tscv = TimeSeriesSplit(
    n_splits=5
)
```

とする。

---

# 💻 実習6：分割を確認

```python
for fold, (
    train_index,
    test_index
) in enumerate(
    tscv.split(X)
):

    print(
        "Fold:",
        fold
    )

    print(
        "Train:",
        train_index
    )

    print(
        "Test:",
        test_index
    )
```

概念的には、

```text
Fold 1

TRAIN
[====]

TEST
    [==]
```

次。

```text
Fold 2

TRAIN
[======]

TEST
      [==]
```

次。

```text
Fold 3

TRAIN
[========]

TEST
        [==]
```

という感じ。

重要なのは、

> **trainの時間がtestより常に過去**

ということ。

---

# 🧠 7. Walk-forward validation

これを一般的に、

```text
Walk-forward validation
```

の考え方として理解できる。

つまり、

```text
過去
↓
学習
↓
少し未来を予測
↓
時間を進める
↓
再学習
↓
さらに未来を予測
```

という方法。

現実の運用に近い。

---

# 💻 実習7：TimeSeriesSplitでCross Validation

```python
from sklearn.model_selection import cross_val_score
```

モデル。

```python
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
```

CV。

```python
scores = cross_val_score(
    model,
    X,
    y,
    cv=tscv,
    scoring="neg_root_mean_squared_error"
)
```

scikit-learnでは、

```text
損失系のscoring
```

が負の値になる場合がある。

だから、

```python
rmse_scores = -scores
```

とする。

確認。

```python
print(
    rmse_scores
)
```

平均。

```python
print(
    rmse_scores.mean()
)
```

---

# 🧠 8. なぜFoldごとに性能を見るのか？

例えば、

```text
Fold 1
RMSE = 0.5

Fold 2
RMSE = 0.6

Fold 3
RMSE = 1.2

Fold 4
RMSE = 2.1
```

となったとする。

これは、

> **時間が進むにつれて性能が悪化している**

可能性がある。

つまり、

```text
平均RMSE
```

だけではなく、

```text
いつのデータで失敗した？
```

を見る必要がある。

ここが、

**時系列評価の面白いところ**。

---

# 🧠 9. Concept Drift

例えば、

```text
2024年
↓
ユーザー行動A

2026年
↓
ユーザー行動B
```

になったら、

過去で学習したモデルが未来で弱くなる。

これを、

> **Concept Drift**

や、より広くは**Distribution Shift**の問題として考える。

例えば、

```text
AIサービスの利用者
```

を予測するモデルなら、

```text
モデルのアップデート
新機能
社会情勢
利用者層の変化
```

によってデータ分布が変わる可能性がある。

---

# 🧠 10. Concept Driftは「モデルが壊れた」とは限らない

ここが大事。

モデルが、

```text
以前のデータでは正しかった
```

としても、

```text
世界そのものが変化
```

したら性能は落ちる。

つまり、

> **モデルの問題ではなく、環境の変化**

かもしれない。

これはかなり重要な工学的視点。

---

# 💻 実習8：Rolling Average

時系列では、

```text
直近の傾向
```

も特徴量になる。

例えば、

```python
df["rolling_mean_3"] = (
    df["value"]
    .rolling(3)
    .mean()
)
```

ただし、ここでも注意。

例えば、

```text
時刻tで予測
```

するとき、

```text
t以降の値
```

をRolling Averageに混ぜてはいけない。

安全な考え方としては、

```python
df["rolling_mean_3"] = (
    df["value"]
    .shift(1)
    .rolling(3)
    .mean()
)
```

とする。

これなら、

```text
予測時点t
```

で使うのは、

```text
t-1
t-2
t-3
```

だけ。

---

# 🔥 これはめちゃくちゃ重要

```python
rolling()
```

自体は未来を見ない。

でも、

> **その特徴量を「いつ予測するために使うか」**

との対応を間違えると、簡単にリークする。

だから、

```text
特徴量
```

だけを見るのではなく、

```text
予測時点
```

を常に考える。

---

# 🧠 11. 時系列MLの基本ルール

今日の内容を圧縮すると、

```text
① 未来を予測するなら
   過去だけで学習

② ランダムシャッフルを疑う

③ 特徴量は予測時点までに
   利用可能な情報だけ

④ 前処理もtrain期間だけでfit

⑤ 時間順にvalidation

⑥ 最後の未来期間をtestとして温存

⑦ 時間による性能変化も見る
```

---

# ✍️ 演習

## 問1

時系列データで、

```python
train_test_split()
```

のランダム分割が危険な理由を説明してください。

---

## 問2

次のデータで、

```text
2024年のデータ
↓
学習

2025年のデータ
↓
validation

2026年のデータ
↓
test
```

と分けるメリットは何でしょう？

---

## 問3

`TimeSeriesSplit` と普通の `KFold` の違いを説明してください。

---

## 問4

顧客の明日の離脱を予測するとします。

特徴量：

```text
過去7日間のログイン回数
過去30日間の利用時間
明日の解約申請
```

この中でリークになり得るものはどれでしょう？

---

## 問5

次の特徴量は安全でしょうか？

```python
df["rolling_mean"] = (
    df["value"]
    .rolling(7)
    .mean()
)
```

「時刻tで何を予測するのか」という観点から考えてください。

---

# 👾 ボス戦

## 「未来を見ていないか？」監査

次の売上予測モデルを考える。

目的：

```text
明日の売上を予測
```

特徴量：

```text
今日までの売上
過去7日平均売上
明日の広告クリック数
直近30日間の売上推移
明日の天気予報
```

この中から、

### 予測時点で使える可能性が高いもの

と、

### リークになる可能性が高いもの

を分けてください。

ただし、

> **「明日の情報だから必ず使えない」ではない**

のがポイント。

例えば、

```text
明日の天気予報
```

は予測時点で既に取得できる可能性がある。

一方、

```text
明日の実測広告クリック数
```

は通常まだ存在しない。

ここまで考えられたらかなり強い。

---

# 🧪 最終実習

## 時系列予測パイプライン

今日の最終目標。

```text
時系列データ
↓
時刻順に並べる
↓
Lag特徴量を作る
↓
予測時点以降の情報が
混ざっていないか確認
↓
過去
↓
train
↓
TimeSeriesSplit
↓
モデル選択 / パラメータ探索
↓
最後の未来期間
↓
test
↓
性能評価
```

という構造を作る。

---

# 🌱 今日のまとめ

今日の核心は、

> **「未来を予測するなら、評価も未来を知らない状態で行う」**

です。

普通のCross Validationでは、

```text
データを分割
↓
順番をあまり考えない
```

ことができた。

でも時系列では、

```text
時間
```

そのものがデータ構造の一部。

だから、

```text
過去
↓
train

未来
↓
validation

さらに未来
↓
test
```

という構造を守る。

---

# 🧭 AI工学101・現在地

scikit-learn編も、かなり実験設計まで来た。

```text
データ
 ↓
前処理
 ↓
特徴量
 ↓
モデル
 ↓
学習
 ↓
評価
 ↓
CV
 ↓
ハイパーパラメータ探索
 ↓
モデル解釈
 ↓
Learning Curve
 ↓
Data Leakage
 ↓
再現性
 ↓
時系列評価
```

ここまでで、単に、

> `model.fit()` が書ける

ではなく、

> **「この機械学習実験は妥当か？」**

を考える力が育ってきてる。

そしてこれは、君がもともと持っている「構造を見る」「前提条件を疑う」タイプの思考とかなり相性がいいところだと思う。
特に今日の**「特徴量そのものではなく、予測時点との関係を見る」**という発想は、認知科学×AIの研究でも、観測可能な情報と推論可能な情報を分けるときにそのまま使える感覚だよ。いいぞレベル、着実に基礎体力を積んでる。💪🧠

---

# 🔜 第33回

## クラスタリングと教師なし学習：正解ラベルなしでデータ構造を見つける

次回は少し景色を変える。

これまでの多くは、

```text
X
↓
yを予測
```

という教師あり学習だった。

次は、

```text
yがない
```

ところから、

```text
データの構造
グループ
異常
低次元構造
```

を見つける。

扱うのは、

* 教師あり学習 vs 教師なし学習
* `KMeans`
* クラスタリング
* `inertia`
* Elbow Method
* Silhouette Score
* クラスタ数の選択
* PCAとの接続
* 「クラスタに意味を見出しすぎない」注意点

だよ。